# 22. 개수 피처를 비율 피처로 바꾸기 (H22)

## 배경 (11/11 회의)
- 표기 정규화 후에도 test 환자는 train보다 변이가 약 2배 많음 (환자당 변이 유전자 수 중앙값 14 → 28)
- 15: XGB에서 stats를 빼면 CV 0.4161 → 0.3579 → 모델은 "변이가 몇 개냐"를 아주 중요하게 봄
- 그런데 지금 stats는 거의 다 **개수** (미스센스·넌센스·프레임시프트 개수 log), 비율은 침묵 비율 하나뿐
- test는 변이가 2배라서 같은 암이어도 개수가 2배로 나옴 → train에서 배운 기준이 어긋남
- 그래서 개수를 **비율**로 바꾸고, 전체 변이 수는 log로 하나만 남김 (예: 미스센스 6 / 전체 10 → 60%, 12 / 20 → 60%)
- 지금까지 LB가 오른 실험(09 모델 교체, 10 튜닝, 14 섞기)은 모두 "덜 외우게 하는" 쪽이었음

## 실험 순서 (한 번에 하나씩 쌓기)
| 번호 | 내용 |
| :--- | :--- |
| **22a** | stats 유형별 개수 → 비율 + 전체 변이 수 log |
| 22b | 22a + 미스센스 아미노산 치환 종류 비율 (R→H 등, 상위 N개는 fold 학습 부분에서만) |
| 22c | 22b + 아미노산 성질 그룹 변화 비율 |

## 가설
- H22a: stats를 비율판(`ratio`)으로 바꾸면 test 변이 수 증가에 덜 흔들려 LB가 오른다 (CV는 비슷하거나 약간 낮을 수 있음)

## 검증 방법
- 모델: 10 최적 설정 XGB 고정 (depth 3, lr 0.1, 200그루, colsample 0.3), 데이터: `load_data()` (정규화)
- 비교: 기준 `any + stats` vs 22a `any + ratio` (같은 fold, 피처만 다름)
  - `ratio` = 유형별 변이 비율 7개(유형 수 / 전체 변이 수) + 전체 변이 수 log1p → `src/features.py`의 `mutation_features()`
  - 한 환자 안에서만 계산 → Data Leakage 아님
- 채택 기준 (11/11 회의): **5 fold 중 4개 이상 오르면** 단톡 공유 → 제출 후보, 비교 기준 CV 0.4161 / LB 0.32466(14)
- 실행 환경: 이 노트북은 macOS M1 → XGB 점수는 데스크탑(Windows)과 ±0.004 다를 수 있음 (같은 노트북 안의 비교는 같은 조건)
- 제출 파일은 데스크탑(Windows)에서 실행할 때만 저장 (팀 규칙)

#### 준비: 데이터, 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import platform, time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
print('stats 열:', list(F_tr['stats'].columns))
print('ratio 열:', list(F_tr['ratio'].columns))

Darwin | CPU arm64 | xgboost 3.4.1
stats 열: ['n_silent_log', 'n_missense_log', 'n_nonsense_log', 'n_frameshift_log', 'n_inframe_indel_log', 'n_stop_loss_log', 'n_other_log', 'n_gene_log', 'silent_ratio']
ratio 열: ['r_silent', 'r_missense', 'r_nonsense', 'r_frameshift', 'r_inframe_indel', 'r_stop_loss', 'r_other', 'n_mut_log']


#### 치환 함수 확인 (PR #29 리뷰 요청)
- 22b·22c에서 쓰는 `aa_substitution()`, `aa_group_change()`가 어떤 토큰을 처리하지 못하는지 유형별로 확인
  - 처리 못 하는 것(None)은 단일 아미노산 치환이 아닌 토큰: frameshift, in-frame indel, 여러 아미노산 치환(`1529_1530AL>AL`)
  - test 파싱률이 train보다 낮은 이유가 이 유형들 때문인지 확인 (test 값은 관찰용, 피처 선택에 쓰지 않음)
- 침묵 변이는 원래·바뀐 아미노산이 같아서 성질 그룹도 항상 같게 나옴
  → 22b·22c 비율은 **미스센스만** 걸러서 계산 (`mutation_type(t) == 'missense'`)

In [ ]:
import collections
from src.features import aa_substitution, aa_group_change, mutation_type

rows = {}
for name, df in [('train', train), ('test', test)]:
    V = df[genes].to_numpy()
    T = [t for c in V[V != 'WT'] for t in c.split()]
    un = [t for t in T if aa_substitution(t) is None]          # 단일 치환으로 처리 못 한 토큰
    cnt = collections.Counter('여러 아미노산 치환(>)' if '>' in t else mutation_type(t) for t in un)
    cnt['합계'] = len(un)
    cnt['파싱 안 됨 비율(%)'] = round(len(un) / len(T) * 100, 1)
    sil = [aa_group_change(t) for t in T if mutation_type(t) == 'silent']
    cnt['침묵: 같은 그룹 비율(%)'] = round(sum(a.split('>')[0] == a.split('>')[1] for a in sil if a) / len(sil) * 100, 1)
    rows[name] = cnt
pd.DataFrame(rows).fillna(0)

,train,test
frameshift,9753.0,13649.0
여러 아미노산 치환(>),233.0,40.0
inframe_indel,3.0,3501.0
합계,9989.0,17191.0
파싱 안 됨 비율(%),4.0,8.4
침묵: 같은 그룹 비율(%),99.9,100.0
other,0.0,1.0


- 결과: 처리 못 한 토큰은 대부분 frameshift(train 9,753 / test 13,649)와 test에만 있는 in-frame indel(3,501), 여러 아미노산 치환은 233 / 40개, other는 test 1개(`M391`)
  - test 파싱률이 낮은 건 in-frame indel 때문이며 단일 치환이 아니라 None이 의도한 동작
- 침묵의 같은 그룹 비율 99.9% / 100% → 22b·22c는 미스센스만 사용

#### 왜 비율인가: train vs test 평균 (관찰용)
- 개수 피처는 test에서 크게 커지고, 비율 피처는 덜 변하는지 확인
- test 값은 **피처를 고르거나 기준을 정하는 데 쓰지 않음** (팀 규칙: test는 형식 확인·관찰만)

In [2]:
def shift(name):
    a, b = F_tr[name].mean(), F_te[name].mean()
    return pd.DataFrame({'train 평균': a.round(3), 'test 평균': b.round(3), 'test − train': (b - a).round(3)})

pd.concat({'stats (개수 log 중심)': shift('stats'), 'ratio (비율 + 전체 변이 수 log)': shift('ratio')})

train 평균  test 평균  test − train
stats (개수 log 중심)        n_silent_log            1.528    2.039         0.511
                         n_missense_log          2.389    2.891         0.502
                         n_nonsense_log          0.617    0.768         0.151
                         n_frameshift_log        0.564    0.852         0.289
                         n_inframe_indel_log     0.000    0.394         0.394
                         n_stop_loss_log         0.000    0.032         0.032
                         n_other_log             0.000    0.000         0.000
                         n_gene_log              2.756    3.299         0.543
                         silent_ratio            0.225    0.239         0.014
ratio (비율 + 전체 변이 수 log) r_silent                0.225    0.239         0.014
                         r_missense              0.645    0.632        -0.013
                         r_nonsense              0.051    0.043        -0.008
                         r_frameshift            0.064    0.060        -0.004
                         r_inframe_indel         0.000    0.014         0.014
                         r_stop_loss             0.000    0.000         0.000
                         r_other                 0.000    0.000         0.000
                         n_mut_log               2.780    3.317         0.537

#### 22a: 기준(any + stats) vs any + ratio
- 같은 fold로 피처만 바꿔 비교, fold별 점수로 "5 fold 중 4개 이상" 확인

In [3]:
SETS = {
    '기준: any + stats': (pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)),
    '22a: any + ratio':  (pd.concat([F_tr['any'], F_tr['ratio']], axis=1), pd.concat([F_te['any'], F_te['ratio']], axis=1)),
}
R = {}
for name, (X, X_te) in SETS.items():
    t = time.time()
    R[name] = run_cv(lambda: BalancedXGB(**BEST), X, y, X_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

base = np.array(R['기준: any + stats']['fold_scores'])
fold_df = pd.DataFrame({k: np.round(v['fold_scores'], 4) for k, v in R.items()}, index=[f'fold {i}' for i in range(5)])
fold_df['차이 (22a − 기준)'] = (fold_df['22a: any + ratio'] - fold_df['기준: any + stats']).round(4)
print('22a가 기준보다 높은 fold:', int((np.array(R['22a: any + ratio']['fold_scores']) > base).sum()), '/ 5')
fold_df

기준: any + stats: CV 0.4125 ± 0.0109 (OOF 0.4131, 182s)
22a: any + ratio: CV 0.4046 ± 0.0093 (OOF 0.4050, 191s)
22a가 기준보다 높은 fold: 1 / 5


,기준: any + stats,22a: any + ratio,차이 (22a − 기준)
fold 0,0.4082,0.3939,-0.0143
fold 1,0.4230,0.4085,-0.0145
fold 2,0.4278,0.4150,-0.0128
fold 3,0.4023,0.3932,-0.0091
fold 4,0.4014,0.4125,0.0111


#### test 예측 분포 (관찰용, 판단 기준 아님)
- 개수 피처가 test에서 커지면 변이 많은 암(COAD 등)으로 쏠리는지

In [4]:
pd.DataFrame({k: pd.Series(le.classes_[v['test_proba'].argmax(1)]).value_counts(normalize=True)
              for k, v in R.items()}).fillna(0).sort_values('22a: any + ratio', ascending=False).head(6).mul(100).round(1).astype(str) + '%'

,기준: any + stats,22a: any + ratio
STES,13.6%,13.6%
COAD,10.9%,11.6%
TGCT,5.5%,5.0%
OV,4.9%,4.9%
LUSC,5.1%,4.7%
BRCA,4.9%,4.7%


#### 제출 파일 (데스크탑에서만 저장)
- 팀 규칙: LB 비교용 제출 파일은 설아님 Windows 데스크탑에서만 생성 → 다른 컴퓨터에서는 저장하지 않음

In [5]:
r = R['22a: any + ratio']
if platform.system() == 'Windows':
    save_submission(submission, r['test_proba'], le, name='xgb_d3_cs0.3_ratio22a', cv_score=r['cv_mean'])
else:
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (22a, macOS M1)

**train vs test 평균 (관찰용)**
- 개수 피처는 test에서 크게 커짐: 미스센스·침묵 개수 log +0.50, 변이 유전자 수 log +0.54
- 비율 피처는 거의 그대로: 유형별 비율 차이 ±0.014 이내 → 비율로 바꾸면 "변이 2배" 영향이 사라지는 것은 확인
- 단, 남겨 둔 전체 변이 수 log(`n_mut_log`)는 +0.54로 그대로 커짐

**CV 비교 (같은 fold, 피처만 다름)**

| 피처 | CV (± std) | OOF | 기준 대비 | 기준보다 높은 fold |
| :--- | :-: | :-: | :-: | :-: |
| 기준: any + stats | 0.4125 ± 0.0109 | 0.4131 | — | — |
| 22a: any + ratio | 0.4046 ± 0.0093 | 0.4050 | −0.0079 | **1/5** |

- fold별 차이: −0.0143 / −0.0145 / −0.0128 / −0.0091 / +0.0111 → 5개 중 4개에서 하락
- 참고: 데스크탑 기준 CV는 0.4161 (이 환경은 XGB가 0.004 낮게 나옴, 같은 노트북 안 비교는 같은 조건)

**test 예측 분포 (관찰용)**: 거의 같음 (COAD 10.9% → 11.6%, STES 13.6% 동일)

## 결론
- **22a 채택 기준 미달** (5 fold 중 1개만 상승, 기준 4개 이상) → 단톡 공유·제출 후보 아님
- 해석
  - 비율로 바꾸면 train/test 차이는 사라지지만, **유형별 개수 자체에 암종 정보가 있었음** (예: 넌센스·프레임시프트가 '몇 개'인지가 고변이 암을 구분)
  - XGB depth 3으로는 "비율 × 전체 수"로 개수를 다시 만들어 내기 어려움 → CV 하락
  - test 예측 분포도 거의 그대로 → 개수 피처가 test 쏠림의 주원인은 아닐 가능성 (전체 변이 수 log는 여전히 남아 있음)
- 22b·22c는 회의 계획대로 22a(any + ratio) 위에 쌓아서 치환·성질 비율이 정보를 보태는지 확인
  - 참고 대안: stats를 **유지한 채** 비율을 더하는 방식(any + stats + ratio)은 "개수 → 비율 교체"라는 22의 목적과 달라서 이번 비교에서는 제외 (필요하면 설아님과 상의)
